In [1]:
# train.py

# Standard library
import argparse
import glob
import json
import os
import shutil
import sys
from dataclasses import dataclass
from datetime import datetime
from functools import partial
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

# Data processing
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset

# Hugging Face datasets and evaluation
import evaluate
from datasets import (
    Dataset as HFDataset,
    concatenate_datasets,
    load_dataset,
)

# Hugging Face Transformers
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

# Training and parameter-efficient fine-tuning
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model

# Experiment tracking
import wandb

In [2]:
odf = pd.read_csv("full_babe_with_cats.csv")
odf['text'] = odf['text'].astype(str).fillna("")
odf["followed_news_outlets_clean"] = (
    odf["followed_news_outlets"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace('"', "", regex=False)
)
odf["num_foll_news_outlet"] = odf["followed_news_outlets_clean"].apply(
    lambda x: len([item for item in x.split(",") if item.strip() != ""])
)

odf["num_age"] = pd.cut(odf["age"],bins=[-np.inf, 18, 30, 45, 55, 65, np.inf],labels=False)

odf["num_poli_idealogy"] = pd.cut(odf["political_ideology"],
                                 bins=[-np.inf, -6, -1, 1, 5, np.inf],
                                 labels=False)

odf["num_foll_news_outlet"] = pd.cut(odf["num_foll_news_outlet"],
                                          bins=[-np.inf, 1, 2, 3, 4, 6, np.inf],
                                          labels=False)


odf["num_news_check_frequency"] = odf["news_check_frequency"].map({
    "Never": 0,
    "Very rarely": 1,
    "Several times per month": 2,
    "Several times per week": 3,
    "Every day": 4,
    "Several times per day": 5,
})

def bin_education(edu):
    edu = str(edu).strip()
    if edu in {"Graduate work"}:
        return 5
    if edu in {"BA", "Bachelor", "Bachelors", "Bachelor's degree", "Bachelor’s degree"}:
        return 4
    if edu in {"Associate degree"}:
        return 3
    if edu in {"Some college", "Vocational or technical school"}:
        return 2
    if edu == "High school graduate":
        return 1
    return 0

odf["num_education"] = odf["education"].apply(bin_education)
odf['num_gender'] = odf['gender'].map({
    'Male': 1, 'male': 1,
    'Female': 0, 'female': 0,
}).fillna(2).astype(int)
odf["label_binary"] = odf["label_bias"].str.startswith("Biased").astype(int)

def binary_entropy(x):
    p = x.mean()
    if p == 0 or p == 1:
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

# odf['text'].nunique #49733
# odf['sentence_id'].nunique #49733 -same

entropy_df = (
    odf.groupby("sentence_id")["label_binary"]
      .agg(
          n="count",
          p_1="mean",
          entropy=binary_entropy
      )
      .reset_index()
)
odf = odf.merge(entropy_df[["sentence_id", "entropy"]], on="sentence_id", how="left")

/scratch/slurm-3164687/ipykernel_3454842/2991592148.py:1: DtypeWarning: Columns (0: survey_completed) have mixed types. Specify dtype option on import or set low_memory=False.
  odf = pd.read_csv("full_babe_with_cats.csv")


In [3]:
#embeds

cat_cardinalities = {
    "words": 11267,
    "factual": 3,
    "group_id": 85,
    "type": 3,
    "topic": 14,
    "outlet": 8,
    "age": 54,
    "gender": 3,
    "education": 8,
    "native_english_speaker": 3,
    "political_ideology": 21,
    "followed_news_outlets": 551,
    "news_check_frequency": 6,
    "num_news_check_frequency": 6,
    "num_foll_news_outlet": 6,
    "num_poli_idealogy": 6,
    "num_age": 6,
    "num_gender": 3,
    "num_education": 6,
    
}



out_cats = ["num_age", "num_gender", "num_education", "num_poli_idealogy", "num_news_check_frequency", "num_foll_news_outlet"]
in_cats = [k for k, card in cat_cardinalities.items() if k not in ["num_age", "num_gender", "num_education", "num_poli_idealogy", "num_news_check_frequency", "num_foll_news_outlet"]]
cat_embs_info = {}
for k, card in cat_cardinalities.items():
    if k not in odf.columns:
        continue
    emb_dim = max(2, int(card ** 0.5) + 1) #sqrt(cardinality)) + 1
    emb_dim = min(50, emb_dim) #but not more than 50 - cough cough words
    cat_embs_info[k] = {
        "cardinality": card, 
        "emb_dim": emb_dim,
        "out_cat": k in out_cats,
        "in_cat": k in in_cats}




class MTLDataset(TorchDataset):
    def __init__(self, 
                 df, 
                 cat_info, 
                 tokenizer,
                 zero_cats = False,
                 max_length=128):
        
        df = df.reset_index(drop=True).copy()
        self.texts = df["text"].tolist()
     
        for col in list(cat_info.keys()):
            df[col] = df[col].astype('category')
            try:
                df[col] = df[col].cat.codes
            except:
                print(col)

        cat_ids = df[list(cat_info.keys())].values   #[N, num_cat_vars]
        cat_ids = cat_ids.astype(int)

        if zero_cats:
            cat_ids = np.zeros_like(cat_ids)

    
        self.cat_ids = cat_ids
        self.labels = df["label_binary"].to_numpy(dtype=np.float32)
        self.entrops = df["entropy"].to_numpy(dtype=np.float32)
        self.out_cat_names = [k for k, v in cat_info.items() if v.get("out_cat", False)]
        self.out_cat_ids = {name: df[name].values for name in self.out_cat_names}
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        cats = self.cat_ids[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "x_cats": torch.tensor(cats, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
            "entrops": torch.tensor(self.entrops[idx], dtype=torch.float),
            
            "target_cats": torch.tensor(
                [self.out_cat_ids[name][idx] for name in self.out_cat_names],
                dtype=torch.long
            ),
        }
        return item
    
    


In [4]:
ds = MTLDataset(train_df, cat_embs_info, tokenizer, max_length=128, zero_cats=False)

# raw array the Dataset stored
print("array dtype/shape:", ds.entrops.dtype, ds.entrops.shape)
print("array nan count:", np.isnan(ds.entrops).sum(), "of", len(ds.entrops))
print("array stats:", ds.entrops.min(), ds.entrops.max(), ds.entrops.mean())

# what __getitem__ actually hands back
item = ds[0]
print("entrops[0]:", item["entrops"], "| isnan:", torch.isnan(item["entrops"]).item())

# spot-check a few more positions
for i in [0, 1, 100, len(ds)-1]:
    print(i, "->", ds[i]["entrops"].item())

NameError: name 'train_df' is not defined

In [83]:
# odf['entropy'].head()
# odf['text'].nunique #49733
# odf.shape = 49734
# odf['sentence_id'].nunique #49733 -same

print("odf rows:", len(odf), "| unique sids:", odf['sentence_id'].nunique())
print("entropy_df rows:", len(entropy_df))
print("sid dtype odf:", odf['sentence_id'].dtype, "| entropy_df:", entropy_df['sentence_id'].dtype)
# how many odf sids actually exist in entropy_df?
print("matched:", odf['sentence_id'].isin(entropy_df['sentence_id']).mean())
# and is entropy itself NaN inside entropy_df, before the merge?
print("entropy_df entropy NaN:", entropy_df['entropy'].isna().sum())


# odf[(odf['dataset'] == 'MBIC') & (odf['sentence_id'].notna())].shape
# # odf['dataset'].value_counts()

# #mbic = 17775/17775
# #sg1 = 13597/13597
# #sg2 = 18362/18362 
# # odf[(odf['dataset'] == 'MBIC')].shape
odf[odf['sentence_id'].isna()][['dataset', 'sentence_id', 'text']] #-49734
# odf[odf['sentence_id'].isna()]['dataset'].value_counts()
# odf.shape

# odf['text'].nunique #49734
# odf['sentence_id'].nunique() #33659 -same


odf rows: 49734 | unique sids: 3696
entropy_df rows: 3696
sid dtype odf: str | entropy_df: str
matched: 1.0
entropy_df entropy NaN: 0


,dataset,sentence_id,text


In [5]:
class ambiProbe(nn.Module):

    
    def __init__(self, 
                cats_info, 
                llm_pretrained_name, 
                parameters
                ):
        super().__init__()

        self.condEntropyWeight = parameters.get('condEntropyWeight')

        self.llm = AutoModel.from_pretrained(
            llm_pretrained_name,
            output_hidden_states=True
        )
        for p in self.llm.parameters():
            p.requires_grad = False
        
        self.llm.eval()

                

        d_model = self.llm.config.hidden_size

        late_fusion_width = d_model + 1 #plus 1 for the label
        self.linear_post_fusion = nn.Linear(late_fusion_width, d_model)

        
        self.shared_trunk = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model // 4),
        )
        
        self.out_cat_names = [k for k, v in cats_info.items() if v.get("out_cat", False)]
        self.probe_heads = nn.ModuleDict({
            name: nn.Linear(d_model // 4, cats_info[name]["cardinality"])
            for name in self.out_cat_names
        })

    def forward(
        self,
        input_ids, #embedding of input text
        attention_mask,
        x_cats,
        labels, # [B] binary silver labels
        target_cats,  # [B, out_cats] numerical features
        entrops = None #[B] entropies per sentence
    ):

        with torch.no_grad():
            outputs = self.llm(input_ids=input_ids, attention_mask=attention_mask)
        #uncollate here instead of building a collator for the dataset?
        target_cats_dict = {}
        for idx, name in enumerate(self.out_cat_names):
            target_cats_dict[name] = target_cats[:, idx]

        last_hidden = outputs.last_hidden_state # [B, L, d_model]
        mask = attention_mask.unsqueeze(-1)            # [B, L, 1]
        hidden_masked = last_hidden * mask             # [B, L, d_model]
        sum_hidden = hidden_masked.sum(dim=1)          # [B, d_model]
        lengths = mask.sum(dim=1).clamp(min=1)         # [B, 1]
        h_text = sum_hidden / lengths                  # [B, d_model]
        

        h_fused = torch.cat([h_text, labels.unsqueeze(-1).float()], dim=-1) # [B, d_model + 1]
        h_proj = self.linear_post_fusion(h_fused)    # [B, d_model]
        h_shared = self.shared_trunk(h_proj)         # [B, shared_hdim]

        

        logits = {name: head(h_shared) for name, head in self.probe_heads.items()}

        for name in self.probe_heads:
            target = target_cats_dict[name]
            logit = logits[name]
            n_classes = logit.shape[-1]


            bad = (target < 0) | (target >= n_classes)
            if bad.any():
                bad_vals = target[bad].detach().cpu().tolist()[:20]
                raise ValueError(
                    f"Bad targets for head {name}: {bad_vals}; "
                    f"valid range is [0, {n_classes - 1}]"
                )


                
        losses = {name: F.cross_entropy(logits[name], target_cats_dict[name], reduction="none") # returns [B] — one loss per row
                  for name in self.probe_heads}
        #stack the piecewise losses
        losses = torch.stack(list(losses.values())) #makes a cats x B
        #sum on the vertical
        losses = losses.sum(dim=0) #now a [1 x B]
        weighted_entrops = entrops.to(losses.device).float()  * self.condEntropyWeight
        cond_losses = losses * weighted_entrops
            
        # total_loss = cond_losses.sum()
        total_loss = cond_losses.sum() / weighted_entrops.sum().clamp(min=1e-8)


        return {
            "loss": total_loss,
            "piecewise_losses": losses,
            "out_logits" : logits
        }
        

# TRAINING
## Tiny Llama

In [6]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "TinyLlama_probe_3epoch_standard"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
parameters = {'condEntropyWeight': 1.0}

# if os.path.exists('./model/probes/TinyLlama_probe/model.safetensors'):
if 1==2:
    tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0", parameters)
    from safetensors.torch import load_file
    tl_probe.load_state_dict(load_file('./model/probes/TinyLlama_probe_3epoch_standard/model.safetensors'))
    tl_probe.to(device)  # if you're on GPU
    tl_probe.eval()
else:
    tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0", parameters)
    # ffdf = odf.sample(1000)
    ffdf = odf.copy()
    train_df, test_df = train_test_split(
        ffdf, 
        test_size=0.2, 
        random_state=42, 
        shuffle=True
    )
    train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                               max_length=128, zero_cats=False)
    eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                               max_length=128, zero_cats=False)
    
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        learning_rate=5e-5,
        num_train_epochs=3,
        weight_decay=0.01,
        save_total_limit=1,
        report_to='wandb',
        seed=42,
    )
    
    trainer = Trainer(
        model=tl_probe,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        compute_metrics=None,
    )
    
    trainer.train()
    trainer.save_model(output_dir)

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /ihome/xli/sek188/.netrc.
wandb: Currently logged in as: sek188 (teamMaverick) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

#### Tiny Llaam Rand Cats

In [9]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "TinyLlama_probe_3epoch_randCats"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
parameters = {'condEntropyWeight': 1.0}

ffdf = odf.copy()
rng = np.random.default_rng(42)
perm = rng.permutation(len(ffdf))
for col in out_cats:
    ffdf[col] = ffdf[col].to_numpy()[perm]

tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0", parameters)
train_df, test_df = train_test_split(
    ffdf, 
    test_size=0.2, 
    random_state=42, 
    shuffle=True
)
train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
    
    
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    report_to='wandb',
    seed=42,
)

trainer = Trainer(
    model=tl_probe,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=None,
)

trainer.train()
trainer.save_model(output_dir)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss
1,6.691200,6.709219
2,6.656200,6.696422
3,6.567600,6.687558


In [ ]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "TinyLlama_probe_7epoch_standard"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
parameters = {'condEntropyWeight': 1.0}

tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0", parameters)

ffdf = odf.copy()
# ffdf = odf.sample(100)
train_df, test_df = train_test_split(
    ffdf, 
    test_size=0.2, 
    random_state=42, 
    shuffle=True
)
train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
    
    
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=7,
    weight_decay=0.01,
    save_total_limit=1,
    report_to='wandb',
    seed=42,
)

trainer = Trainer(
    model=tl_probe,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=None,
)

trainer.train()
trainer.save_model(output_dir)

eval/loss,▁▆█▆▆▄▄
eval/runtime,▁▁▁▁▁▁█
eval/samples_per_second,██████▁
eval/steps_per_second,██████▁
train/epoch,▁▂▃▅▆▇██
train/global_step,▁▂▃▅▆▇██
eval/loss,10.07955
eval/runtime,0.434
eval/samples_per_second,46.081
eval/steps_per_second,6.912
total_flos,0


Epoch,Training Loss,Validation Loss
1,6.705400,6.784556
2,6.715200,6.717883
3,6.705000,6.691912
4,6.609100,6.683404
5,6.547700,6.693762


In [ ]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "TinyLlama_probe_12epoch_standard"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
parameters = {'condEntropyWeight': 1.0}

ffdf = odf.copy()
rng = np.random.default_rng(42)
perm = rng.permutation(len(ffdf))
for col in out_cats:
    ffdf[col] = ffdf[col].to_numpy()[perm]

tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0", parameters)
train_df, test_df = train_test_split(
    ffdf, 
    test_size=0.2, 
    random_state=42, 
    shuffle=True
)
train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
    
    
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    report_to='wandb',
    seed=42,
)

trainer = Trainer(
    model=tl_probe,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=None,
)

trainer.train()
trainer.save_model(output_dir)

In [15]:
# len(train_dataset.texts) = 9946/
# odf.shape = 49734

(49734, 33)

#### Qwen

In [ ]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "Qwen_probe"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
model = ambiProbe(cat_embs_info, "Qwen/Qwen-7B")
# ffdf = odf.sample(1000)
ffdf = odf.copy()
train_df, test_df = train_test_split(
    ffdf, 
    test_size=0.8, 
    random_state=42, 
    shuffle=True
)
train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)


training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    # report_to='wandb',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=None,
)

trainer.train()
trainer.save_model(output_dir)

#### Deberta

In [ ]:
PROBE_WEIGHTS_DIR = './model/probes/'
run_name   = "magpie-pt_probe"
output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

wandb.init(
    project="bias-ambivalence-mtl",
    name=run_name,
    config={},
)
tokenizer = AutoTokenizer.from_pretrained("mediabiasgroup/magpie-pt")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
model = ambiProbe(cat_embs_info, "mediabiasgroup/magpie-pt")
# ffdf = odf.sample(1000)
ffdf = odf.copy()
train_df, test_df = train_test_split(
    ffdf, 
    test_size=0.8, 
    random_state=42, 
    shuffle=True
)
train_dataset = MTLDataset(train_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)
eval_dataset  = MTLDataset(test_df, cat_embs_info, tokenizer, 
                           max_length=128, zero_cats=False)


training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    # report_to='wandb',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=None,
)

trainer.train()
trainer.save_model(output_dir)
   

# TESTING
#### Tiny Llama

In [ ]:
import torch
from collections import Counter
from torch.utils.data import DataLoader
from tqdm import tqdm

model.eval()
device = next(model.parameters()).device
out_cat_names = eval_dataset.out_cat_names

marginal_acc = {}
for name in out_cat_names:
    counts = Counter(eval_dataset.out_cat_ids[name].tolist())
    marginal_acc[name] = max(counts.values()) / sum(counts.values())

correct = {name: 0 for name in out_cat_names}
total = 0
loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)

with torch.no_grad():
    for batch in tqdm(loader, desc="Evaluating probe"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        x_cats         = batch["x_cats"].to(device)
        target_cats    = batch["target_cats"].to(device)
        
        out = model(input_ids, attention_mask, x_cats, labels, target_cats)
        
        bsz = input_ids.size(0)
        total += bsz
        for cat_idx, name in enumerate(out_cat_names):
            preds = out["out_logits"][name].argmax(dim=-1)
            golds = target_cats[:, cat_idx]
            correct[name] += (preds == golds).sum().item()

print(f"\n{'Variable':<25} {'Probe':>8}  {'Marginal':>10}  {'Lift':>8}")
print("─" * 55)
for name in out_cat_names:
    probe_acc = correct[name] / total
    marg_acc  = marginal_acc[name]
    lift      = probe_acc - marg_acc
    print(f"{name:<25} {probe_acc:>8.3f}  {marg_acc:>10.3f}  {lift:>+8.3f}")

print("─" * 55)
global_probe = sum(correct.values()) / (total * len(out_cat_names))
global_marg  = sum(marginal_acc.values()) / len(out_cat_names)
print(f"{'GLOBAL':<25} {global_probe:>8.3f}  {global_marg:>10.3f}  {global_probe - global_marg:>+8.3f}")

In [ ]:
import torch
import torch.nn.functional as F
import random

def bar(p, width=20):
    filled = int(round(p * width))
    return "█" * filled + "·" * (width - filled)

model.eval()
device = next(model.parameters()).device
out_cat_names = eval_dataset.out_cat_names
indices = random.sample(range(len(eval_dataset)), 3)

for idx in indices:
    item = eval_dataset[idx]
    text = eval_dataset.texts[idx]
    
    input_ids     = item["input_ids"].unsqueeze(0).to(device)
    attention_mask = item["attention_mask"].unsqueeze(0).to(device)
    labels        = item["labels"].unsqueeze(0).to(device)
    x_cats        = item["x_cats"].unsqueeze(0).to(device)
    target_cats   = item["target_cats"].unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(input_ids, attention_mask, x_cats, labels, target_cats)
    
    text_display = text if len(text) <= 140 else text[:140] + "…"
    print("\n" + "═" * 75)
    print(f"  Text:        {text_display}")
    print(f"  Bias label:  {int(item['labels'].item())}")
    print("─" * 75)
    
    for cat_idx, cat_name in enumerate(out_cat_names):
        logits = out["out_logits"][cat_name].squeeze(0)
        probs = F.softmax(logits, dim=-1)
        gold = int(item["target_cats"][cat_idx].item())
        pred = int(probs.argmax().item())
        correct = pred == gold
        
        flag = "✓" if correct else "✗"
        print(f"\n  {cat_name}   gold={gold}  pred={pred}  p(pred)={probs[pred].item():.2f}  p(gold)={probs[gold].item():.2f}  {flag}")
        
        for j, p in enumerate(probs.tolist()):
            if j == gold and j == pred:
                marker = "← gold/pred"
            elif j == gold:
                marker = "← gold"
            elif j == pred:
                marker = "← pred"
            else:
                marker = ""
            print(f"    {j}  {p:.2f}  {bar(p)}  {marker}")
    print()